In this notebook we will load the Hoefling et al., 2024 data and build dataloaders using the JAX backend.

It mirrors the data-loading parts of `training_hoefling_2024.ipynb` and is meant to be run cell by cell.

# Imports

In [1]:
import logging
import os

import hydra
import matplotlib
import numpy as np

from openretina.data_io.base import compute_data_info
from openretina.data_io.hoefling_2024.dataloaders import natmov_dataloaders_v2
from openretina.data_io.hoefling_2024.responses import filter_responses, make_final_responses
from openretina.data_io.hoefling_2024.stimuli import movies_from_pickle
from openretina.utils.file_utils import get_local_file_path
from openretina.utils.h5_handling import load_h5_into_dict
from openretina.utils.misc import CustomPrettyPrinter
from openretina.utils.plotting import numpy_to_mp4_video

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)  # to display logs in jupyter

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

pp = CustomPrettyPrinter(indent=4, max_lines=30)

Let's import also the global config file for this model using hydra.

In [2]:
with hydra.initialize(config_path=os.path.join("..", "configs"), version_base="1.3"):
    cfg = hydra.compose(config_name="hoefling_2024_core_readout_low_res.yaml")

# Loading data

The first step in loading data is determining from where it will be fetched / stored.

Let's see how this is handled in the configs:

In [3]:
pp.pprint(cfg.paths)

{   'cache_dir': '${oc.env:OPENRETINA_CACHE_DIRECTORY}',
    'data_dir': '${paths.cache_dir}/data/',
    'load_model_path': None,
    'log_dir': '.',
    'movies_path': 'https://huggingface.co/datasets/open-retina/open-retina/blob/main/euler_lab/hoefling_2024/stimuli/rgc_natstim_18x16_joint_normalized_2024-01-11.zip',
    'output_dir': '${hydra:runtime.output_dir}',
    'responses_path': 'https://huggingface.co/datasets/open-retina/open-retina/resolve/main/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.zip'}


The config contains the path from where files will be downloaded, and also requires the `cache_dir` to be set by the user: this is the directory where the data will be stored on download.

When using the training script, if cache_dir is not set by the user in the config files or somewhere in the script, this will fall back to the `OPENRETINA_CACHE_DIRECTORY` environment variable, which by default points to `~/openretina_cache`.

If set, the `cache_dir` is also what the package will use in place of the default openretina cache folder. Let's set both here:

In [4]:
your_chosen_root_folder = "."  # Change this with your desired path.

cfg.paths.cache_dir = your_chosen_root_folder

# We will also overwrite the output directory for the logs/model to the local folder.
cfg.paths.log_dir = your_chosen_root_folder
cfg.paths.output_dir = your_chosen_root_folder

os.environ["OPENRETINA_CACHE_DIRECTORY"] = "./openretina_cache"

## Stimuli

Loading of the stimuli is achieved, in the training script, via:
```
movies_dict = hydra.utils.call(cfg.data_io.stimuli)
```

Let's unpack it here.

In [5]:
pp.pprint(cfg.data_io.stimuli)

{   '_convert_': 'object',
    '_target_': 'openretina.data_io.hoefling_2024.stimuli.movies_from_pickle',
    'file_path': {   '_convert_': 'object',
                     '_target_': 'openretina.utils.file_utils.get_local_file_path',
                     'cache_folder': '${paths.cache_dir}',
                     'file_path': '${paths.movies_path}'}}


Essentially, using the `get_local_file_path` function, if `file_path` is not a local file, it will be downloaded to the cache folder and read from there.

In [6]:
movies_path = get_local_file_path(file_path=cfg.paths.movies_path, cache_folder=cfg.paths.data_dir)

movies_dict = movies_from_pickle(movies_path)

2026-02-14 16:18:38,052 - INFO - Fetching file list for open-retina/open-retina...
2026-02-14 16:18:39,320 - INFO - Found target file at /home/samuel/GitRepos/open-retina/notebooks/data/euler_lab/hoefling_2024/stimuli/rgc_natstim_18x16_joint_normalized_2024-01-11.pkl


In [7]:
pp.pprint(movies_dict)

MoviesTrainTestSplit(train=numpy.ndarray(shape=(2, 16200, 18, 16)),
                     test_dict={   'test': numpy.ndarray(shape=(2, 750, 18, 16))},
                     stim_id=None,
                     random_sequences=numpy.ndarray(shape=(108, 20)),
                     norm_mean=nan,
                     norm_std=nan)


Let us also visualize a few seconds of the training video:

In [8]:
numpy_to_mp4_video(movies_dict.train[:, :300, ...])

## Responses

In the training script, responses are loaded through:

```
neuron_data_dict = hydra.utils.call(cfg.data_io.responses)
```

Let's unpack it here.

In [9]:
pp.pprint(cfg.data_io.responses)

{   '_convert_': 'object',
    '_target_': 'openretina.data_io.hoefling_2024.responses.make_final_responses',
    'data_dict': {   '_convert_': 'object',
                     '_target_': 'openretina.data_io.hoefling_2024.responses.filter_responses',
                     'all_responses': {   '_convert_': 'object',
                                          '_target_': 'openretina.utils.h5_handling.load_h5_into_dict',
                                          'file_path': {   '_convert_': 'object',
                                                           '_target_': 'openretina.utils.file_utils.get_local_file_path',
                                                           'cache_folder': '${paths.cache_dir}',
                                                           'file_path': '${paths.responses_path}'}},
                     'cell_types_list': '${quality_checks.cell_types_list}',
                     'chirp_qi': '${quality_checks.chirp_qi}',
                     'classifier_confid

While this may look complex, it effectively amounts to resolving a few intermediate steps in loading the data, and should be read from the inside out.

When written more simply, it is equivalent to the following:

In [11]:
responses_path = get_local_file_path(file_path=cfg.paths.responses_path, cache_folder=cfg.paths.data_dir)

responses_dict = load_h5_into_dict(file_path=responses_path)

filtered_responses_dict = filter_responses(responses_dict, **cfg.quality_checks)

final_responses = make_final_responses(filtered_responses_dict, response_type="natural")

2026-02-14 16:22:38,299 - INFO - Fetching file list for open-retina/open-retina...
2026-02-14 16:22:38,737 - INFO - Downloading euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.zip...


euler_lab/hoefling_2024/responses/rgc_na(…):   0%|          | 0.00/815M [00:00<?, ?B/s]

2026-02-14 16:39:05,018 - INFO - Extracting data/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.zip...
2026-02-14 16:39:11,280 - INFO - Extracted single file to /home/samuel/GitRepos/open-retina/notebooks/data/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.h5.


Loading HDF5 file contents:   0%|          | 0/2077 [00:00<?, ?item/s]

Original dataset contains 7863 neurons over 67 fields
 ------------------------------------ 
Dropped 0 fields that did not contain the target cell types (67 remaining)
Overall, dropped 3034 neurons of non-target cell types (-38.59%).
 ------------------------------------ 
Dropped 0 fields with quality indices below threshold (67 remaining)
Overall, dropped 980 neurons over quality checks (-20.29%).
 ------------------------------------ 
Dropped 0 fields with classifier confidences below 0.25
Overall, dropped 705 neurons with classifier confidences below 0.25 (-18.32%).
 ------------------------------------ 
 ------------------------------------ 
Final dataset contains 3144 neurons over 67 fields
Total number of cells dropped: 4719 (-60.02%)


Upsampling natural spikes traces to get final responses.:   0%|          | 0/67 [00:00<?, ?it/s]

And here is how the final responses will be organised:

In [12]:
pp.pprint(final_responses)

{   'session_1_ventral1_20200226': ResponsesTrainTestSplit(train=numpy.ndarray(shape=(80, 16200)),
                                                           test_dict={   'test': numpy.ndarray(shape=(80, 750))},
                                                           test_by_trial=numpy.ndarray(shape=(80, 750, 3)),
                                                           test_by_trial_dict={   'test': numpy.ndarray(shape=(80, 750, 3))},
                                                           stim_id='natural',
                                                           session_kwargs={   'eye': 'left',
                                                                              'group_assignment': numpy.ndarray(shape=(80,)),
                                                                              'roi_ids': numpy.ndarray(shape=(80,)),
                                                                              'roi_mask': numpy.ndarray(shape=(64, 64)),
                  

# Creating dataloaders (JAX backend)

The corresponding code in `train.py` is:
```
dataloaders = hydra.utils.instantiate(
        cfg.dataloader,
        neuron_data_dictionary=neuron_data_dict,
        movies_dictionary=movies_dict,
    )
```

For the JAX backend, make sure JAX is installed (`pip install jax jaxlib`). JAX also requires `num_workers=0` in the PyTorch DataLoader, which is the default.

In [ ]:
pp.pprint(cfg.dataloader)

In [13]:
dataloaders = natmov_dataloaders_v2(
    neuron_data_dictionary=final_responses,
    movies_dictionary=movies_dict,
    allow_over_boundaries=True,
    batch_size=128,
    train_chunk_size=50,
    validation_clip_indices=cfg.dataloader.validation_clip_indices,
    array_backend="jax",
)

Creating movie dataloaders:   0%|          | 0/67 [00:00<?, ?it/s]

INFO:2026-02-14 16:54:20,605:jax._src.xla_bridge:834: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-02-14 16:54:20,605 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


In [18]:
input,target = next(iter(dataloaders["train"]["session_1_ventral1_20200226"]))

In [24]:
input.shape

(128, 2, 50, 18, 16)

In [14]:
pp.pprint(dataloaders)

{   'test': {   'session_1_ventral1_20200226': torch.utils.data.DataLoader(Dataset: MovieDataSet with 80 neuron responses to a movie of shape [2, 750, 18, 16].),
                'session_1_ventral1_20200528': torch.utils.data.DataLoader(Dataset: MovieDataSet with 42 neuron responses to a movie of shape [2, 750, 18, 16].),
                'session_1_ventral1_20200707': torch.utils.data.DataLoader(Dataset: MovieDataSet with 74 neuron responses to a movie of shape [2, 750, 18, 16].),
                'session_1_ventral1_20201021': torch.utils.data.DataLoader(Dataset: MovieDataSet with 32 neuron responses to a movie of shape [2, 750, 18, 16].),
                'session_1_ventral1_20201030': torch.utils.data.DataLoader(Dataset: MovieDataSet with 40 neuron responses to a movie of shape [2, 750, 18, 16].),
                'session_1_ventral1_20210929': torch.utils.data.DataLoader(Dataset: MovieDataSet with 48 neuron responses to a movie of shape [2, 750, 18, 16].),
                'session_1_v

Let's also compute `data_info`, which is used to initialise certain model components and to save important metadata about stimuli and responses within the model.

In [15]:
data_info = compute_data_info(neuron_data_dictionary=final_responses, movies_dictionary=movies_dict)

pp.pprint(data_info)

{   'input_shape': (2, 18, 16),
    'mean_activity_dict': {   'session_1_ventral1_20200226': torch.Tensor(shape=[80]),
                              'session_1_ventral1_20200528': torch.Tensor(shape=[42]),
                              'session_1_ventral1_20200707': torch.Tensor(shape=[74]),
                              'session_1_ventral1_20201021': torch.Tensor(shape=[32]),
                              'session_1_ventral1_20201030': torch.Tensor(shape=[40]),
                              'session_1_ventral1_20210929': torch.Tensor(shape=[48]),
                              'session_1_ventral1_20210930': torch.Tensor(shape=[26]),
                              'session_1_ventral2_20200302': torch.Tensor(shape=[41]),
                              'session_1_ventral2_20200707': torch.Tensor(shape=[56]),
                              'session_1_ventral2_20201021': torch.Tensor(shape=[39]),
                              'session_1_ventral2_20201022': torch.Tensor(shape=[42]),
           